# Analise eleitoral de Divinopolis - 2026

**Disciplina:** Ferramentas para Data Science - UFSJ

**Municipio:** Divinopolis, Minas Gerais

**Objetivo:** analisar a distribuicao espacial dos votos
presidenciais nos dois turnos da eleicao de 2026.

O trabalho utiliza Pandas, GeoPandas e Folium, com dados
obtidos do Portal de Dados Abertos do TSE.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

RAIZ = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
sys.path.insert(0, str(RAIZ))

DADOS = RAIZ / "dados" / "processados"
votos = pd.read_csv(
    DADOS / "votos_divinopolis_2026.csv",
    sep=";", encoding="utf-8-sig"
)
locais = pd.read_csv(
    DADOS / "locais_divinopolis_2026.csv",
    sep=";", encoding="utf-8-sig"
)
resumo = pd.read_csv(
    DADOS / "resumo_secoes_divinopolis_2026.csv",
    sep=";", encoding="utf-8-sig"
)
display(votos.head())


,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_TURNO,NR_ZONA,NR_SECAO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS
0,MG,44458,DIVINÓPOLIS,1,102,1,13,LUIZ INÁCIO LULA DA SILVA,74
1,MG,44458,DIVINÓPOLIS,1,102,1,14,RENAN ANTONIO FERREIRA DOS SANTOS,6
2,MG,44458,DIVINÓPOLIS,1,102,1,22,FLAVIO NANTES BOLSONARO,198
3,MG,44458,DIVINÓPOLIS,1,102,1,27,CLARIANA ZACARKIM BARAO,1
4,MG,44458,DIVINÓPOLIS,1,102,1,30,ROMEU ZEMA NETO,1


## Metodologia

Foram utilizadas duas bases oficiais do TSE: votacao por
secao eleitoral e eleitorado por local de votacao.

Primeiro, os dados foram filtrados para Divinopolis/MG,
municipio de codigo 44458.

Posteriormente, os registros foram associados utilizando
municipio, turno, zona e secao eleitoral.

Os votos nominais foram separados dos votos brancos e
nulos para calcular os percentuais de votos validos.

Como diversas secoes compartilham o mesmo local de
votacao, os votos foram agregados por coordenada geografica
para definir as cores dos mapas.

As regioes foram construidas utilizando diagramas de
Voronoi no sistema UTM 23S (EPSG:32723), com limite
visual de aproximadamente 1.200 metros dos locais.

**Limitacao:** essas regioes representam proximidade
geografica, nao limites eleitorais oficiais ou locais de
residencia dos eleitores.


In [2]:
chaves = [
    "CD_MUNICIPIO", "NR_TURNO",
    "NR_ZONA", "NR_SECAO"
]

print("Registros de votos:", len(votos))
print("Registros de locais:", len(locais))
print("Secoes por turno:")
print(resumo.groupby("NR_TURNO").size())

assert not resumo.duplicated(chaves).any()
assert not resumo[["latitude", "longitude"]].isna().any().any()


Registros de votos: 6144
Registros de locais: 2016
Secoes por turno:
NR_TURNO
1    501
2    501
dtype: int64


In [3]:
# Os scripts abaixo contem o processamento completo.
# Com os ZIPs oficiais em dados/brutos, pode-se
# reproduzir a filtragem e a integracao desde o inicio.

from filtrar_divinopolis import main as filtrar
from integrar_dados import main as integrar

# filtrar()
# integrar()

# Os arquivos processados ja incluidos no repositorio
# permitem executar as analises sem baixar os ZIPs.


In [4]:
votos_validos = votos[
    votos["NR_VOTAVEL"].between(10, 94)
].copy()

resultados = (
    votos_validos.groupby(
        ["NR_TURNO", "NR_VOTAVEL", "NM_VOTAVEL"],
        as_index=False
    )["QT_VOTOS"].sum()
)

totais = (
    votos_validos.groupby("NR_TURNO")["QT_VOTOS"]
    .sum()
)

resultados["percentual"] = resultados.apply(
    lambda r: 100 * r.QT_VOTOS / totais.loc[r.NR_TURNO],
    axis=1
)
display(resultados.sort_values(
    ["NR_TURNO", "QT_VOTOS"],
    ascending=[True, False]
))


,NR_TURNO,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,percentual
4,1,22,FLAVIO NANTES BOLSONARO,71478,53.982735
0,1,13,LUIZ INÁCIO LULA DA SILVA,49403,37.310908
11,1,70,AUGUSTO JORGE CURY,4568,3.449917
1,1,14,RENAN ANTONIO FERREIRA DOS SANTOS,3219,2.431104
10,1,55,RONALDO RAMOS CAIADO,2214,1.672092
8,1,30,ROMEU ZEMA NETO,1277,0.964436
12,1,80,SAMARA MARTINS DA SILVA FEITOSA,130,0.098181
5,1,27,CLARIANA ZACARKIM BARAO,34,0.025678
2,1,16,HERTZ DA CONCEICAO DIAS,31,0.023412
3,1,21,EDMILSON SILVA COSTA,21,0.015860


In [5]:
zonas = (
    votos_validos.groupby(
        ["NR_TURNO", "NR_ZONA", "NR_VOTAVEL"],
        as_index=False
    )["QT_VOTOS"].sum()
)

zonas["total_zona"] = zonas.groupby(
    ["NR_TURNO", "NR_ZONA"]
)["QT_VOTOS"].transform("sum")

zonas["percentual"] = (
    100 * zonas.QT_VOTOS / zonas.total_zona
)
display(zonas[zonas.NR_VOTAVEL.isin([13, 22])])


,NR_TURNO,NR_ZONA,NR_VOTAVEL,QT_VOTOS,total_zona,percentual
0,1,102,13,27291,77278,35.315355
4,1,102,22,43497,77278,56.286395
13,1,103,13,22112,55131,40.108106
17,1,103,22,27981,55131,50.753659
26,2,102,13,34434,81066,42.476501
27,2,102,22,46632,81066,57.523499
28,2,103,13,28060,57968,48.406017
29,2,103,22,29908,57968,51.593983


## Mapa 1 - Flavio Bolsonaro, primeiro turno

### O fato

Flavio Bolsonaro recebeu 71.478 votos, equivalentes a
53,98% dos 132.409 votos validos de Divinopolis.

Na zona 102, alcancou 56,29% dos votos validos,
enquanto na zona 103 obteve 50,75%.

Entre os locais com maior percentual para Flavio,
destacam-se o Instituto Nossa Senhora do Sagrado
Coracao, com 68,91%, e o Colegio Roberto Carneiro,
com 66,17%, ambos no Centro.

Os resultados demonstram variacao espacial no apoio
ao candidato, com diferencas importantes entre locais.

### O design

Foi utilizada uma paleta sequencial azul, com cinco
cores de referencia e interpolacao continua de 0% a
100%. Portanto, nao foram definidas classes discretas.

A opacidade de 70% permite observar as vias e outras
referencias do mapa-base OpenStreetMap.

A representacao utiliza poligonos GeoJSON, Voronoi,
tooltips e popups interativos do Folium.


## Mapa 2 - Lula, primeiro turno

### O fato

Lula recebeu 49.403 votos, correspondentes a 37,31%
dos votos validos no municipio.

Na zona 102, obteve 35,32% dos votos validos;
na zona 103, obteve 40,11%.

Entre os locais com maior percentual para Lula,
destacam-se a Escola Municipal Benjamim Constant,
no Buritis, com 56,55%, e a Escola Municipal Joao
Gontijo da Fonseca, no Primavera, com 53,69%.

A comparacao com o mapa de Flavio evidencia que o
apoio aos candidatos nao se distribuiu uniformemente
pelos locais de votacao.

### O design

Foi utilizada uma escala sequencial vermelha,
com cinco cores de referencia e interpolacao
continua entre 0% e 100%.

A transparencia de 70% preserva a leitura do
mapa-base e permite identificar referencias
geograficas proximas aos locais de votacao.

Os tooltips exibem resultados por secao, enquanto os
popups permitem consultar informacoes adicionais.


In [6]:
from gerar_mapas_primeiro_turno import main as gerar_primeiro
gerar_primeiro()



MAPAS DO 1o TURNO
Secoes eleitorais: 501
Coordenadas distintas: 72
FLAVIO NANTES BOLSONARO - 71,478 votos: C:\Users\cassi\Documents\VScode\6 Periodo\TDS\eleicoes2026_divinopolis\Trabalho_TSE\mapas\primeiro_turno\primeiro_turno_candidato_22.html
LUIZ INÁCIO LULA DA SILVA - 49,403 votos: C:\Users\cassi\Documents\VScode\6 Periodo\TDS\eleicoes2026_divinopolis\Trabalho_TSE\mapas\primeiro_turno\primeiro_turno_candidato_13.html
Mapas gerados. Abra os arquivos HTML em um navegador.


## Mapa 3 - Comparacao do segundo turno

### O fato

No segundo turno, Flavio recebeu 76.540 votos
(55,05%) e Lula recebeu 62.494 votos (44,95%).

A diferenca entre os candidatos foi de 14.046 votos,
menor que a diferenca de 22.075 votos observada
no primeiro turno.

A participacao de Lula aumentou 7,64 pontos
percentuais e a de Flavio aumentou 1,07 ponto
percentual em relacao aos votos validos do
primeiro turno.

Na zona 103, a votacao foi mais equilibrada:
51,59% para Flavio e 48,41% para Lula.

Entre os locais com maior percentual para Lula
estao a Escola Municipal Benjamim Constant
(64,80%) e a Escola Municipal Antonieta Fonseca
(63,97%).

Para Flavio, destacam-se o Instituto Nossa Senhora
do Sagrado Coracao (70,97%) e o Colegio Roberto
Carneiro (68,94%).

Esses resultados evidenciam diferencas territoriais
entre os locais de votacao. Entretanto, nao permitem
identificar transferencias individuais de votos.

### O design

Foi utilizada uma paleta divergente:
azul para vantagem de Flavio, vermelho para
vantagem de Lula e tons claros para equilibrio.

A escala utiliza cinco cores de referencia, com
interpolacao continua de 0% a 100% de votos validos
para Lula. O ponto neutro corresponde a 50%.

A opacidade das regioes e de 73%.

O mapa utiliza Folium, poligonos GeoJSON,
controle de camadas, tooltips e popups.


In [7]:
from gerar_mapa_segundo_turno import main as gerar_segundo
gerar_segundo()



MAPA DO 2o TURNO
Secoes eleitorais: 501
Coordenadas distintas: 72
Lula: 62,494 votos validos
Flavio: 76,540 votos validos
Total validos: 139,034
Mapa salvo em: C:\Users\cassi\Documents\VScode\6 Periodo\TDS\eleicoes2026_divinopolis\Trabalho_TSE\mapas\segundo_turno\segundo_turno_comparativo.html


In [8]:
# Confere os arquivos HTML produzidos.
mapas = sorted((RAIZ / "mapas").rglob("*.html"))
for mapa in mapas:
    print(mapa.relative_to(RAIZ))


mapas\primeiro_turno\primeiro_turno_candidato_13.html
mapas\primeiro_turno\primeiro_turno_candidato_22.html
mapas\segundo_turno\segundo_turno_comparativo.html


## Conclusao

A analise demonstrou que Flavio Bolsonaro liderou
a votacao presidencial de Divinopolis nos dois turnos.

Embora Lula tenha aumentado sua participacao percentual
entre os turnos, Flavio permaneceu com a maior parcela
dos votos validos no municipio.

As diferencas entre as zonas 102 e 103 e entre os locais
de votacao mostram a importancia da representacao
espacial para interpretar dados eleitorais.

Os mapas interativos permitiram examinar os resultados
em diferentes escalas de detalhe, associando os valores
estatisticos a pontos geograficos.

A principal limitacao metodologica esta no uso de
regioes aproximadas de Voronoi. Por isso, a analise
descreve resultados dos locais de votacao e nao
preferencias de todos os moradores de determinada area.

### Fontes

- Tribunal Superior Eleitoral (TSE):
  https://dadosabertos.tse.jus.br/
- Votacao por secao eleitoral - 2026.
- Eleitorado por local de votacao - 2026.
